In [153]:
import csv
data=[]
res=[]
with open("data.csv") as f:
    reader=csv.reader(f)
    for row in reader:
        data.append(row)
        res.append(row[-1])
headers=data[0]
headers.append("Play tennis")
data=data[1:]
res=res[1:]

In [178]:
nData=len(data)
nAttr=len(headers)-1
print("headers=",headers)
for i in range(nData):
    print(data[i],res[i])

headers= ['Outlook', 'Temperature', 'Humidity', 'Wind', 'PlayTennis', 'Play tennis']
['Sunny', 'Hot', 'High', 'Weak', 'No'] No
['Sunny', 'Hot', 'High', 'Strong', 'No'] No
['Overcast', 'Hot', 'High', 'Weak', 'Yes'] Yes
['Rain', 'Mild', 'High', 'Weak', 'Yes'] Yes
['Rain', 'Cool', 'Normal', 'Weak', 'Yes'] Yes
['Rain', 'Cool', 'Normal', 'Strong', 'No'] No
['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'] Yes
['Sunny', 'Mild', 'High', 'Weak', 'No'] No
['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'] Yes
['Rain', 'Mild', 'Normal', 'Weak', 'Yes'] Yes
['Sunny', 'Mild', 'Normal', 'Strong', 'Yes'] Yes
['Overcast', 'Mild', 'High', 'Strong', 'Yes'] Yes
['Overcast', 'Hot', 'Normal', 'Weak', 'Yes'] Yes
['Rain', 'Mild', 'High', 'Strong', 'No'] No


In [155]:
from math import log, exp
def ENTROPY(s):
    yes=s.count('Yes')
    no=s.count('No')
    if yes==0 or no==0:
        return 0
    tot=yes+no
    x=(yes/tot)*log(yes/tot,2)
    y=(no/tot)*log(no/tot,2)
    return -(x+y)
    

In [156]:
## example
ENTROPY(['Yes','No','No'])

0.9182958340544896

In [157]:
def subtables(data, attrColNumber, shudDeleteThatCol):
    dic={}
    for row in data:
        key=row[attrColNumber]
        thatRow=row.copy()
        if shudDeleteThatCol:
            del thatRow[attrColNumber]
        dic.setdefault(key,[]).append(thatRow)
    return list(dic.keys()),dic
        

In [158]:
## example
possibleValuesOfThatCol,dic=subtables(data,1,True)
print("possble values of 1st col",possibleValuesOfThatCol)
print("dict values:")
for key in dic:
    print(key,dic.get(key),"\n")

possble values of 1st col ['Hot', 'Mild', 'Cool']
dict values:
Hot [['Sunny', 'High', 'Weak', 'No'], ['Sunny', 'High', 'Strong', 'No'], ['Overcast', 'High', 'Weak', 'Yes'], ['Overcast', 'Normal', 'Weak', 'Yes']] 

Mild [['Rain', 'High', 'Weak', 'Yes'], ['Sunny', 'High', 'Weak', 'No'], ['Rain', 'Normal', 'Weak', 'Yes'], ['Sunny', 'Normal', 'Strong', 'Yes'], ['Overcast', 'High', 'Strong', 'Yes'], ['Rain', 'High', 'Strong', 'No']] 

Cool [['Rain', 'Normal', 'Weak', 'Yes'], ['Rain', 'Normal', 'Strong', 'No'], ['Overcast', 'Normal', 'Strong', 'Yes'], ['Sunny', 'Normal', 'Weak', 'Yes']] 



In [169]:
def INFO_GAIN(data,attrColNumber):
    sum_of_entropies_of_each_possible_value_of_this_attribute=0
    possibleValuesOfThatAttribute,dic=subtables(data,attrColNumber,False)
    for v in possibleValuesOfThatAttribute:
        subsetsItAppeared=dic[v]
        nSubsetsItAppeared=len(subsetsItAppeared)
        x=(nSubsetsItAppeared/nData)*ENTROPY([row[-1] for row in subsetsItAppeared])
        print("entropy of ",headers[attrColNumber],"->",v,"=",x)
        sum_of_entropies_of_each_possible_value_of_this_attribute+=x
    overal_entropy_of_dataset=ENTROPY([row[-1] for row in data])
    return overal_entropy_of_dataset-sum_of_entropies_of_each_possible_value_of_this_attribute

In [171]:
print("info gain of ",headers[0],"is",INFO_GAIN(data,0))

entropy of  Outlook -> Sunny = 0.3467680694480959
entropy of  Outlook -> Overcast = 0.0
entropy of  Outlook -> Rain = 0.3467680694480959
info gain of  Outlook is 0.2467498197744391


In [237]:
class Node:
    def __init__(self,attrName):
        self.attrName=attrName
        self.yOrNo=""
        self.children=[]

In [270]:
def buildTree(currData,currHeader):
    x=[row[-1] for row in currData]
    if len(set(x))==1:## either just ['Yes'] or ['No']
        leaf=Node("")
        leaf.yOrNo=x[0]
        return leaf
    ## Step1: calc info gain of each attr
    all_attr_info_gains={}
    max_IG=0
    its_colNumber=-1;
    for colNumber in range(nAttr-1):
        attrName=currHeader[colNumber]
        ig=INFO_GAIN(currData,colNumber)
        all_attr_info_gains[attrName]=ig
        if(ig>max_IG):
            max_IG=ig
            its_colNumber=colNumber
    print("all atr info gains= ",all_attr_info_gains)

    # Step2: choose that attr as node tht has max IG
    attr_name=currHeader[its_colNumber]
    print(attr_name, "has max info gain of ",max_IG)
    root=Node(attr_name)

    ## Now recursively calc for other attributes
    rem_atrributes=currHeader[:its_colNumber]+currHeader[its_colNumber+1:]
    print("remaiing attributes are ",rem_atrributes)
    possibleValuesOfThatCol,dic=subtables(currData,its_colNumber,True)
    print(dic)
    for v in possibleValuesOfThatCol:
        root.children.append((v,buildTree(dic[v],rem_atrributes)))
    # print("children",root.children)
    return root;

In [271]:
root=buildTree(data,headers)


entropy of  Outlook -> Sunny = 0.3467680694480959
entropy of  Outlook -> Overcast = 0.0
entropy of  Outlook -> Rain = 0.3467680694480959
entropy of  Temperature -> Hot = 0.2857142857142857
entropy of  Temperature -> Mild = 0.39355535745192405
entropy of  Temperature -> Cool = 0.23179374984546652
entropy of  Humidity -> High = 0.4926140680171258
entropy of  Humidity -> Normal = 0.29583638929116374
entropy of  Wind -> Weak = 0.46358749969093305
entropy of  Wind -> Strong = 0.42857142857142855
all atr info gains=  {'Outlook': 0.2467498197744391, 'Temperature': 0.029222565658954647, 'Humidity': 0.15183550136234136, 'Wind': 0.04812703040826927}
Outlook has max info gain of  0.2467498197744391
remaiing attributes are  ['Temperature', 'Humidity', 'Wind', 'PlayTennis', 'Play tennis']
{'Sunny': [['Hot', 'High', 'Weak', 'No'], ['Hot', 'High', 'Strong', 'No'], ['Mild', 'High', 'Weak', 'No'], ['Cool', 'Normal', 'Weak', 'Yes'], ['Mild', 'Normal', 'Strong', 'Yes']], 'Overcast': [['Hot', 'High', 'Wea

In [268]:
def print_tree(node, level=0):
    if node.yOrNo:
        print("---" * level, node.yOrNo)
        return
    print("---" * level, node.attrName)
    for val, child in node.children:
        print("---" * (level + 1), val)
        print_tree(child, level + 2)


In [269]:
print_tree(root,0)

 Outlook
--- Sunny
------ Humidity
--------- High
------------ No
--------- Normal
------------ Yes
--- Overcast
------ Yes
--- Rain
------ Wind
--------- Weak
------------ Yes
--------- Strong
------------ No
